# D-12 · baseline reproducible y bloqueo de entrenamiento

Este cuaderno NO contiene un entrenador neuronal ejecutable. Incluye extracción, TTF, composición simple, edición, restauración y A/B. CPU; GPU consumida por este cuaderno: 0. No descarga pesos ni instala paquetes. No abrir el examen d10-examen. Solo documentos propios. Un único origen de desarrollo no constituye reserva.

D-12 sigue BLOQUEADA para generación neuronal, LPIPS y evaluación independiente. No lanzar GPU hasta resolver licencias y pesos autorizados, corpus y saldo compartido D-10/D-12 <=20 unidades. Futuro entrenador: padre/maestro/simple/placebo, curvas de predicciones, checkpoints a Drive antes de seguir, olvido y custodio.

In [ ]:
from pathlib import Path
import json, hashlib, sys
ROOT = Path("/content/letra-viva-c3")
PAYLOAD = {'__init__.py': '"""D-12: laboratorio local de reconstrucción tipográfica; sin promoción automática."""\n', 'comun.py': "import hashlib\nimport json\nfrom pathlib import Path\n\n\ndef sha256(path):\n    h=hashlib.sha256()\n    with Path(path).open('rb') as f:\n        for b in iter(lambda:f.read(1<<20),b''):h.update(b)\n    return h.hexdigest()\n\n\ndef guardar(path, data):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    with path.open('x',encoding='utf-8') as f:\n        json.dump(data,f,ensure_ascii=False,indent=2,allow_nan=False);f.write('\\n')\n\n\ndef abrir_adn(pdf, adn):\n    data=json.loads(Path(adn).read_text())\n    if data['sha256_origen']!=sha256(pdf):raise ValueError('ADN no corresponde al documento')\n    pages=data.get('paginas',[data])\n    if not isinstance(pages,list):raise ValueError('ADN requiere páginas completas')\n    return data,pages\n\n\ndef validar_documento(doc, propio):\n    if not propio:raise ValueError('Declaración de documento propio/ejemplo autorizado requerida')\n    if doc.needs_pass or doc.get_sigflags()>0:raise ValueError('Documento cifrado o firmado')\n    for page in doc:\n        if any(a.type[0]==13 for a in page.annots() or []):raise ValueError('Documento con sellos')\n        if any(w.field_type==6 for w in page.widgets() or []):raise ValueError('Campo de firma')\n    # Firmas/sellos rasterizados requieren delimitación humana de zonas excluidas.\n", 'fuente.py': '"""Fuente sin plantilla: medoid por carácter y contornos con huecos. No usa Potrace."""\nfrom pathlib import Path\nimport math\nimport shutil\nimport numpy as np\nimport cv2\nimport pymupdf as fitz\nfrom PIL import Image\nfrom fontTools.fontBuilder import FontBuilder\nfrom fontTools.pens.ttGlyphPen import TTGlyphPen\nfrom fontTools.ttLib import TTFont, TTLibError\nfrom .comun import abrir_adn, guardar, sha256, validar_documento\n\n\ndef medoid(samples):\n    # Tamaño normalizado solo para selección; el contorno conserva escala original.\n    small=np.stack([cv2.resize(s[\'mask\'],(32,32),interpolation=cv2.INTER_AREA).ravel()/255 for s in samples])\n    # Distancias por bloques para no construir N×N×1024.\n    distances=np.maximum(0,(small*small).sum(1)[:,None]+(small*small).sum(1)[None,:]-2*small@small.T)\n    return int(np.argmin(distances.sum(1)))\n\n\ndef contorno(mask, em_px, baseline, left=0.):\n    if em_px<=0 or not np.isfinite(em_px):raise ValueError(\'Escala de glifo inválida\')\n    contours,hierarchy=cv2.findContours(mask,cv2.RETR_TREE,cv2.CHAIN_APPROX_SIMPLE)\n    pen=TTGlyphPen(None)\n    if hierarchy is None:return pen.glyph()\n    for i,contour in enumerate(contours):\n        points=contour[:,0,:].astype(float)\n        if len(points)<3 or abs(cv2.contourArea(contour))<1:continue\n        # RETR_TREE conserva niveles: contorno exterior y agujeros en sentidos opuestos.\n        depth=0;parent=hierarchy[0][i][3]\n        while parent>=0:depth+=1;parent=hierarchy[0][parent][3]\n        xy=np.column_stack(((points[:,0]+left)*1000/em_px,(baseline-points[:,1])*1000/em_px))\n        signed=float(np.sum(xy[:,0]*np.roll(xy[:,1],-1)-np.roll(xy[:,0],-1)*xy[:,1]))\n        if (depth%2==0 and signed>0) or (depth%2==1 and signed<0):xy=xy[::-1]\n        pen.moveTo(tuple(xy[0]))\n        for point in xy[1:]:pen.lineTo(tuple(point))\n        pen.closePath()\n    return pen.glyph()\n\n\ndef construir(pdf, adn, salida, *, pagina=1, excluir_spans=(), dpi=300, propio=False,\n              fuente_xref=None):\n    salida=Path(salida)\n    if salida.exists():raise ValueError(\'No sobrescribir fuente\')\n    data,pages=abrir_adn(pdf,adn);page_data=pages[pagina-1]\n    cv2.setNumThreads(1)\n    with fitz.open(pdf) as doc:\n        validar_documento(doc,propio)\n        page=doc[pagina-1]\n        if page.rotation:raise ValueError(\'Normalizar coordenadas de página rotada antes de extraer\')\n        pix=page.get_pixmap(dpi=dpi,alpha=False);raster=np.frombuffer(pix.samples,np.uint8).reshape(pix.height,pix.width,3)\n        source_fonts=[s.get(\'fuente_xref\') for i,s in enumerate(page_data[\'spans\']) if i not in excluir_spans]\n        xrefs=set(source_fonts)\n        if fuente_xref is None:\n            if len(xrefs)!=1:raise ValueError(\'Varios estilos/fuentes: elegir fuente_xref, no mezclarlos\')\n            fuente_xref=next(iter(xrefs))\n        samples={};excluded=0\n        for si,span in enumerate(page_data[\'spans\']):\n            if si in excluir_spans or span.get(\'fuente_xref\')!=fuente_xref:continue\n            if abs(span.get(\'inclinacion\') or 0)>0.01:excluded+=1;continue\n            chars=span.get(\'caracteres\')\n            if not chars:\n                if len(span[\'texto\'])!=1:excluded+=1;continue\n                chars=[dict(c=span[\'texto\'],bbox=span[\'bbox\'],origin=span[\'linea_base\'][\'origen\'])]\n            for ci,char in enumerate(chars):\n                c=char[\'c\']\n                if len(c)!=1 or c.isspace():continue\n                box=np.array(char[\'bbox\'])*dpi/72\n                x0,y0=np.floor(box[:2]).astype(int);x1,y1=np.ceil(box[2:]).astype(int)\n                if x0<0 or y0<0 or x1>pix.width or y1>pix.height or x1<=x0 or y1<=y0:excluded+=1;continue\n                crop=raster[y0:y1,x0:x1];paper=np.percentile(crop.reshape(-1,3),90,axis=0)\n                dist=np.linalg.norm(crop.astype(float)-paper,axis=2)\n                mask=(dist>max(30.,float(dist.max())*.35)).astype(np.uint8)*255\n                if np.count_nonzero(mask)<3:excluded+=1;continue\n                em=span[\'tamano_pt\']*dpi/72\n                baseline=char[\'origin\'][1]*dpi/72-y0\n                left=x0-char[\'origin\'][0]*dpi/72\n                advance=(chars[ci+1][\'origin\'][0]-char[\'origin\'][0])*1000/span[\'tamano_pt\'] if ci+1<len(chars) else (char[\'bbox\'][2]-char[\'origin\'][0])*1000/span[\'tamano_pt\']\n                samples.setdefault(c,[]).append(dict(mask=mask,em_px=em,baseline=baseline,left=left,\n                    advance=max(100,advance),span=si,caracter=ci,bbox=char[\'bbox\']))\n        if not samples:raise ValueError(\'No hay muestras utilizables\')\n        salida.mkdir(parents=True,exist_ok=False)\n        embedded=None;embedding_reason=None\n        if fuente_xref:\n            _,ext,_,blob=doc.extract_font(fuente_xref)\n            if blob:\n                native=salida/f\'incrustada.{ext}\';native.write_bytes(blob)\n                try:\n                    tt=TTFont(native);cmap=tt.getBestCmap() or {}\n                    fs=tt[\'OS/2\'].fsType if \'OS/2\' in tt else 0\n                    if fs & (2|256|512):embedding_reason=\'Restricción de embedding/edición: revisar licencia original\'\n                    elif all(ord(c) in cmap for c in samples):embedded=native.name\n                    else:embedding_reason=\'Subconjunto incrustado sin cobertura Unicode suficiente\'\n                    tt.close()\n                except TTLibError:embedding_reason=\'Fuente CID/CFF sin contenedor sfnt/cmap Unicode reutilizable por este módulo\'\n        selected={c:values[medoid(values)] for c,values in sorted(samples.items())}\n        glyphs={\'.notdef\':TTGlyphPen(None).glyph(),\'space\':TTGlyphPen(None).glyph()}\n        metrics={\'.notdef\':(500,0),\'space\':(300,0)};cmap={32:\'space\'};records=[]\n        for c,s in selected.items():\n            name=f\'uni{ord(c):04X}\';glyphs[name]=contorno(s[\'mask\'],s[\'em_px\'],s[\'baseline\'],s[\'left\'])\n            metrics[name]=(int(round(s[\'advance\'])),int(round(s[\'left\']*1000/s[\'em_px\'])))\n            cmap[ord(c)]=name\n            target=salida/f\'U{ord(c):04X}.png\';Image.fromarray(s[\'mask\']).save(target)\n            records.append(dict(caracter=c,n=len(samples[c]),span=s[\'span\'],indice=s[\'caracter\'],\n                bbox=s[\'bbox\'],mask=target.name,mask_sha256=sha256(target),avance=metrics[name][0]))\n        fb=FontBuilder(1000,isTTF=True);fb.setupGlyphOrder(list(glyphs));fb.setupCharacterMap(cmap)\n        fb.setupGlyf(glyphs);fb.setupHorizontalMetrics(metrics);fb.setupHorizontalHeader(ascent=1200,descent=-400)\n        fb.setupNameTable(dict(familyName=\'Prometeo Documento\',styleName=\'Regular\',\n            uniqueFontIdentifier=\'Prometeo-\'+data[\'sha256_origen\'][:16],fullName=\'Prometeo Documento Regular\',psName=\'PrometeoDocumento-Regular\'))\n        fb.setupOS2(sTypoAscender=1200,sTypoDescender=-400,usWinAscent=1400,usWinDescent=500)\n        fb.setupPost();fb.setupMaxp();fb.save(salida/\'observada.ttf\')\n        report=dict(version=1,sha256_origen=data[\'sha256_origen\'],pagina=pagina,dpi=dpi,\n            fuente_xref=fuente_xref,excluir_spans=list(excluir_spans),muestras=sum(map(len,samples.values())),\n            glifos_observados=len(selected),omitidos=excluded,glifos=records,\n            fuente=embedded or \'observada.ttf\',sha256_fuente=sha256(salida/(embedded or \'observada.ttf\')),\n            incrustada_reutilizable=embedded,incrustada_limitacion=embedding_reason,\n            metodo=\'contornos raster observados; medoid; sin plantilla; no entrenamiento\',\n            licencia_fuente=\'Solo experimento local sobre documento autorizado; no acredita licencia para distribuir fuente derivada\')\n        guardar(salida/\'fuente.json\',report)\n        return report\n\n\ndef componer_faltante(mask_a, mask_b, regla):\n    """Baseline de combinación explícita, sin atribuir identidad al contorno."""\n    if regla not in (\'union\',\'interseccion\'):raise ValueError(\'Regla no soportada\')\n    if mask_a.shape!=mask_b.shape:raise ValueError(\'Alinear glifos antes de combinar\')\n    return np.maximum(mask_a,mask_b) if regla==\'union\' else np.minimum(mask_a,mask_b)\n\n\ndef completar_mayuscula(fuente, salida, *, destino, modelo_minuscula, referencia_altura):\n    """Baseline explícita: reescala los trazos de un glifo similar observado.\n\n    No aprende, no garantiza identidad y no rellena caracteres silenciosamente.\n    """\n    from fontTools.pens.boundsPen import BoundsPen\n    from fontTools.pens.transformPen import TransformPen\n    salida=Path(salida)\n    if salida.exists():raise ValueError(\'No sobrescribir fuente completada\')\n    font=TTFont(fuente);cmap=font.getBestCmap();glyphset=font.getGlyphSet()\n    if len(destino)!=1 or ord(destino) in cmap:raise ValueError(\'Destino debe ser un carácter faltante\')\n    if any(ord(c) not in cmap for c in (modelo_minuscula,referencia_altura)):raise ValueError(\'Faltan glifos de referencia\')\n    bounds=[]\n    for c in (modelo_minuscula,referencia_altura):\n        pen=BoundsPen(glyphset);glyphset[cmap[ord(c)]].draw(pen)\n        if pen.bounds is None:raise ValueError(\'Glifo vacío\')\n        bounds.append(pen.bounds)\n    a,b=bounds;sy=(b[3]-b[1])/(a[3]-a[1]);sx=(b[2]-b[0])/(a[2]-a[0])\n    pen=TTGlyphPen(glyphset);glyphset[cmap[ord(modelo_minuscula)]].draw(TransformPen(pen,(sx,0,0,sy,b[0]-a[0]*sx,b[1]-a[1]*sy)))\n    name=f\'uni{ord(destino):04X}\';order=list(font.getGlyphOrder())\n    font[\'glyf\'][name]=pen.glyph()\n    font.setGlyphOrder(order+[name]);font[\'hmtx\'].metrics[name]=font[\'hmtx\'].metrics[cmap[ord(referencia_altura)]]\n    for table in font[\'cmap\'].tables:\n        if table.isUnicode():table.cmap[ord(destino)]=name\n    font.save(salida);font.close()\n    record=dict(metodo=\'baseline reescala trazos de glifo similar observado\',destino=destino,\n        origen=modelo_minuscula,referencia_altura=referencia_altura,escala=[sx,sy],\n        sha256_padre=sha256(fuente),sha256_salida=sha256(salida),aprendido=False,\n        advertencia=\'Glifo aproximado, no muestra observada ni resultado de modelo\')\n    guardar(salida.with_suffix(\'.json\'),record)\n    return record\n', 'editar.py': '"""Edición raster autorizada, inpainting y escritura con historial reversible."""\nfrom pathlib import Path\nimport json\nimport math\nimport time\nimport cv2\nimport numpy as np\nimport pymupdf as fitz\nfrom PIL import Image\nfrom fontTools.ttLib import TTFont\nfrom skimage.color import lab2rgb, rgb2lab\nfrom skimage.metrics import structural_similarity\nfrom .comun import abrir_adn,guardar,sha256,validar_documento\n\n\ndef caja_px(box,dpi,width,height):\n    if len(box)!=4 or not np.isfinite(box).all():raise ValueError(\'Caja inválida\')\n    x0,y0=map(math.floor,np.asarray(box[:2])*dpi/72)\n    x1,y1=map(math.ceil,np.asarray(box[2:])*dpi/72)\n    if not 0<=x0<x1<=width or not 0<=y0<y1<=height:raise ValueError(\'Caja fuera de página\')\n    return x0,y0,x1,y1\n\n\ndef dentro(outer,inner):return outer[0]<=inner[0] and outer[1]<=inner[1] and inner[2]<=outer[2] and inner[3]<=outer[3]\ndef intersecta(a,b):return a[0]<b[2] and b[0]<a[2] and a[1]<b[3] and b[1]<a[3]\n\n\ndef reponer_region(original, editada, box):\n    out=original.copy();x0,y0,x1,y1=box\n    out[y0:y1,x0:x1]=editada[y0:y1,x0:x1]\n    return out\n\n\ndef editar(pdf, adn, fuente, salida, *, pagina=1, span_id=18, texto=None,\n           autorizado=None, excluidas=(), propio=False, dpi=144, metodo=\'estilo\'):\n    started=time.perf_counter();salida=Path(salida)\n    if salida.exists():raise ValueError(\'Salida existente\')\n    if metodo not in (\'simple\',\'estilo\'):raise ValueError(\'Método desconocido\')\n    if not 72<=dpi<=300:raise ValueError(\'DPI fuera del rango de laboratorio\')\n    data,pages=abrir_adn(pdf,adn);span=pages[pagina-1][\'spans\'][span_id]\n    if \'educational\' in span[\'texto\'].casefold():raise ValueError(\'Marca educativa excluida de edición\')\n    if abs(span.get(\'inclinacion\') or 0)>.01:raise ValueError(\'Aún no admite texto inclinado; no fingir soporte\')\n    original_text=span[\'texto\'];texto=original_text if texto is None else texto\n    if not isinstance(texto,str) or not texto:raise ValueError(\'Texto vacío\')\n    font=fitz.Font(fontfile=str(fuente))\n    if any(c!=\' \' and not font.has_glyph(ord(c)) for c in texto):raise ValueError(\'Glifo faltante: construir baseline explícita o abstenerse\')\n    if autorizado is None or not dentro(autorizado,span[\'bbox\']):raise ValueError(\'Falta región autorizada que contenga el texto\')\n    if any(intersecta(autorizado,r) for r in excluidas):raise ValueError(\'Región intersecta firma/sello/zona excluida\')\n    with fitz.open(pdf) as doc:\n        validar_documento(doc,propio);page=doc[pagina-1]\n        if page.rotation:raise ValueError(\'Página rotada aún no soportada\')\n        pix=page.get_pixmap(dpi=dpi,alpha=False)\n        original=np.frombuffer(pix.samples,np.uint8).reshape(pix.height,pix.width,3).copy()\n        box=caja_px(autorizado,dpi,pix.width,pix.height);x0,y0,x1,y1=box\n        crop=original[y0:y1,x0:x1];paper=np.percentile(crop.reshape(-1,3),90,axis=0)\n        mask=np.zeros(original.shape[:2],np.uint8)\n        chars=span.get(\'caracteres\') or [dict(c=original_text,bbox=span[\'bbox\'],origin=span[\'linea_base\'][\'origen\'])]\n        for char in chars:\n            cb=caja_px(char[\'bbox\'],dpi,pix.width,pix.height);a,b,c,d=cb\n            local=original[b:d,a:c].astype(float)\n            # Máscara de tinta dentro de cajas aprobadas; dilatar antialiasing sin salir de región.\n            foreground=(np.linalg.norm(local-paper,axis=2)>30).astype(np.uint8)*255\n            mask[b:d,a:c]=np.maximum(mask[b:d,a:c],foreground)\n        mask=cv2.dilate(mask,np.ones((3,3),np.uint8))\n        authorized_mask=np.zeros_like(mask);authorized_mask[y0:y1,x0:x1]=255;mask &= authorized_mask\n        if metodo==\'simple\':\n            canvas=original.copy();canvas[y0:y1,x0:x1]=np.round(paper).astype(np.uint8)\n        else:\n            cv2.setNumThreads(1)\n            # Procesar un margen local; no inpaint sobre toda la lámina.\n            margin=8;a=max(0,x0-margin);b=max(0,y0-margin);c=min(pix.width,x1+margin);d=min(pix.height,y1+margin)\n            canvas=original.copy();canvas[b:d,a:c]=cv2.inpaint(original[b:d,a:c],mask[b:d,a:c],3,cv2.INPAINT_TELEA)\n        origins=np.asarray([c[\'origin\'] for c in chars],float)\n        residuals=origins[:,1]-np.median(origins[:,1])\n        measured_lab=np.array(span[\'tinta_lab\'],float)\n        rgb=np.clip(lab2rgb(measured_lab.reshape(1,1,3))[0,0]*255,0,255)\n        original_ink=original[mask>0]\n        ink_noise=float(np.std(original_ink.mean(1))) if len(original_ink) else 0.\n        with fitz.open() as overlay:\n            target=overlay.new_page(width=page.rect.width,height=page.rect.height)\n            writer=fitz.TextWriter(target.rect);size=span[\'tamano_pt\']\n            if texto==original_text and metodo==\'estilo\':\n                for c in chars:writer.append(c[\'origin\'],c[\'c\'],font=font,fontsize=size)\n            else:\n                writer.append(span[\'linea_base\'][\'origen\'],texto,font=font,fontsize=size)\n            if not fitz.Rect(autorizado).contains(writer.text_rect):\n                # Writer rect may include ascender allowance: check actual alpha below.\n                pass\n            writer.write_text(target,color=(0,0,0))\n            rendered=target.get_pixmap(dpi=dpi,alpha=True)\n            rgba=np.frombuffer(rendered.samples,np.uint8).reshape(rendered.height,rendered.width,4)\n            alpha=rgba[:,:,3].astype(float)/255\n        # Clipping silent prohibited: reject if new ink would leave authorization.\n        if np.any((alpha>0)&(authorized_mask==0)):raise ValueError(\'Texto nuevo excede región autorizada\')\n        # Residual de tinta solo en caso estilo; amplitud conservadora medida del núcleo.\n        noise=np.zeros((y1-y0,x1-x0,1))\n        if metodo==\'estilo\' and len(original_ink):\n            core=original_ink[np.linalg.norm(original_ink.astype(float)-paper,axis=1)>np.percentile(np.linalg.norm(original_ink.astype(float)-paper,axis=1),70)]\n            sigma=float(core.std(axis=0).mean()) if len(core)>1 else 0.\n            noise=np.random.default_rng(2610).normal(0,min(sigma,3.),noise.shape)\n        local_alpha=alpha[y0:y1,x0:x1,None]\n        composed=canvas.copy();composed[y0:y1,x0:x1]=np.clip(np.rint(canvas[y0:y1,x0:x1]*(1-local_alpha)+(rgb+noise)*local_alpha),0,255).astype(np.uint8)\n        result=reponer_region(original,composed,box)\n        diff=np.any(result!=original,axis=2);outside=diff&(authorized_mask==0)\n        if np.any(outside):raise AssertionError(\'Modificación fuera de región autorizada\')\n        salida.mkdir(parents=True,exist_ok=False)\n        Image.fromarray(original).save(salida/\'pagina-antes.png\');Image.fromarray(result).save(salida/\'pagina-despues.png\')\n        Image.fromarray(crop).save(salida/\'parche-antes.png\');Image.fromarray(result[y0:y1,x0:x1]).save(salida/\'parche-despues.png\')\n        target=result[y0:y1,x0:x1]\n        metrics=dict(ssim=float(structural_similarity(crop,target,channel_axis=2,data_range=255)),\n            mae_rgb=float(np.abs(crop.astype(float)-target).mean()),lpips=None,\n            lpips_estado=\'NO VERIFICADO: pesos no autorizados ni descargados\',\n            pixeles_fuera_modificados=int(outside.sum()),pixeles_modificados=int(diff.sum()),\n            segundos=time.perf_counter()-started,gpu_unidades=0,coste_usd=None)\n        report=dict(version=1,sha256_origen=data[\'sha256_origen\'],sha256_adn=sha256(adn),\n            sha256_fuente=sha256(fuente),pagina=pagina,span=span_id,region_pt=autorizado,region_px=list(box),\n            dpi=dpi,metodo=metodo,antes=original_text,despues=texto,tamano_pt=size,tinta_lab=measured_lab.tolist(),\n            inclinacion=span.get(\'inclinacion\'),residual_linea_base_pt=residuals.tolist(),\n            textura_tinta_std_observada=ink_noise,region_excluidas=list(excluidas),\n            sha256_antes=sha256(salida/\'pagina-antes.png\'),sha256_despues=sha256(salida/\'pagina-despues.png\'),\n            sha256_parche_antes=sha256(salida/\'parche-antes.png\'),sha256_parche_despues=sha256(salida/\'parche-despues.png\'),\n            metricas=metrics,dominio=\'DESARROLLO_MISMO_ORIGEN\',aprendizaje_pesos=False,\n            limite=\'Inpainting local puede degradar líneas/grano dentro de la región; revisar visualmente\')\n        guardar(salida/\'edicion.json\',report);return report\n', 'restaurar.py': '"""Restauración comprobada del raster, sin alterar el PDF original."""\nimport json\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image\nfrom .comun import sha256\n\n\ndef restaurar(edicion, salida):\n    edicion, salida = Path(edicion), Path(salida)\n    if salida.exists():\n        raise ValueError(\'No sobrescribir restauración\')\n    r = json.loads((edicion/\'edicion.json\').read_text())\n    for name, key in [(\'pagina-despues.png\', \'sha256_despues\'),\n                      (\'parche-antes.png\', \'sha256_parche_antes\'),\n                      (\'pagina-antes.png\', \'sha256_antes\')]:\n        if sha256(edicion/name) != r[key]:\n            raise ValueError(\'Historial o imagen alterados: \'+name)\n    current = np.array(Image.open(edicion/\'pagina-despues.png\').convert(\'RGB\'))\n    before = np.array(Image.open(edicion/\'pagina-antes.png\').convert(\'RGB\'))\n    patch = np.array(Image.open(edicion/\'parche-antes.png\').convert(\'RGB\'))\n    x0,y0,x1,y1 = r[\'region_px\']\n    if patch.shape != (y1-y0,x1-x0,3):\n        raise ValueError(\'Dimensiones del parche incorrectas\')\n    current[y0:y1,x0:x1] = patch\n    if not np.array_equal(current,before):\n        raise ValueError(\'Restauración no coincide píxel a píxel con original\')\n    salida.parent.mkdir(parents=True,exist_ok=True)\n    Image.fromarray(current).save(salida)\n    return dict(pixel_identico=True,sha256_restaurada=sha256(salida),\n                sha256_original_raster=r[\'sha256_antes\'])\n', 'ab.py': '"""A/B local: participante sin clave; custodio separado. Nunca fabrica respuestas."""\nfrom pathlib import Path\nimport argparse\nimport hashlib\nimport json\nimport math\nimport random\nimport shutil\nfrom .comun import guardar,sha256\n\n\ndef preparar(pares, participante, custodio, seed=2610):\n    participante,custodio=Path(participante),Path(custodio)\n    if participante.exists() or custodio.exists():raise ValueError(\'No sobrescribir prueba A/B\')\n    if participante.resolve() in custodio.resolve().parents or custodio.resolve() in participante.resolve().parents:\n        raise ValueError(\'Participante y custodio deben ser carpetas separadas\')\n    participante.mkdir(parents=True);custodio.mkdir(parents=True)\n    rng=random.Random(seed);order=list(range(len(pares)));rng.shuffle(order)\n    public=[];key=[]\n    for n,i in enumerate(order):\n        pair=pares[i];left_original=bool(rng.getrandbits(1))\n        originals=[pair[\'original\'],pair[\'rehecha\']]\n        if not left_original:originals.reverse()\n        names=[]\n        for side,path in zip((\'A\',\'B\'),originals):\n            name=f\'{n:04d}-{side}.png\';shutil.copyfile(path,participante/name);names.append(name)\n        ident=f\'{n:04d}\';public.append(dict(id=ident,A=names[0],B=names[1]))\n        key.append(dict(id=ident,original=\'A\' if left_original else \'B\',sha256_origen=pair[\'sha256_origen\']))\n    session=hashlib.sha256(json.dumps(key,sort_keys=True).encode()).hexdigest()\n    guardar(custodio/\'clave.json\',dict(version=1,sesion=session,seed=seed,pares=key))\n    guardar(participante/\'pares.json\',dict(version=1,sesion=session,pares=public))\n    template=(Path(__file__).parent/\'ab.html\').read_text()\n    # Datos estructurados; escapar \'<\' por si hubiera entradas externas futuras.\n    payload=json.dumps(dict(sesion=session,pares=public)).replace(\'<\',\'\\\\u003c\')\n    (participante/\'index.html\').write_text(template.replace(\'__DATOS__\',payload))\n    return dict(sesion=session,n=len(public),respuestas=0,resultado=\'PENDIENTE_BRUNO\',\n        clave_sha256=sha256(custodio/\'clave.json\'))\n\n\ndef evaluar(respuestas,clave):\n    r=json.loads(Path(respuestas).read_text());k=json.loads(Path(clave).read_text())\n    if r[\'sesion\']!=k[\'sesion\']:raise ValueError(\'Sesión incorrecta\')\n    truth={x[\'id\']:x[\'original\'] for x in k[\'pares\']};answers=r[\'respuestas\']\n    if len({x[\'id\'] for x in answers})!=len(answers):raise ValueError(\'Respuesta duplicada\')\n    if any(x[\'id\'] not in truth or x[\'eleccion\'] not in (\'A\',\'B\') for x in answers):raise ValueError(\'Respuesta inválida\')\n    n=len(answers)\n    if n==0:return dict(n=0,acierto=None,resultado=\'NO EVALUABLE\')\n    hits=sum(x[\'eleccion\']==truth[x[\'id\']] for x in answers);p=hits/n;z=1.959963984540054\n    den=1+z*z/n;center=(p+z*z/(2*n))/den\n    half=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))/den\n    return dict(n=n,total=len(truth),aciertos=hits,acierto=p,wilson95=[center-half,center+half],\n        origenes=len({x[\'sha256_origen\'] for x in k[\'pares\']}),meta_puntual=p<=.55,\n        cumple_prerregistro=False,nota=\'Requiere reserva independiente, >=100 palabras y >=10 páginas; revisar agrupación por origen\')\n\n\nif __name__==\'__main__\':\n    ap=argparse.ArgumentParser(description=__doc__);ap.add_argument(\'respuestas\');ap.add_argument(\'clave\')\n    a=ap.parse_args();print(json.dumps(evaluar(a.respuestas,a.clave),indent=2))\n', 'ab.html': '<!doctype html><html lang="es"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Letra viva · prueba A/B</title>\n<style>body{background:#f4f2ec;color:#233b30;font:17px system-ui;margin:50px auto;max-width:1100px;padding:20px}h1{font-size:36px}#pares{display:flex;gap:24px}.opcion{background:white;padding:28px;border:1px solid #cbd4c9;border-radius:8px;flex:1;text-align:center}img{width:100%;height:220px;object-fit:contain;image-rendering:auto}button{background:#255642;color:white;border:0;padding:14px 24px;border-radius:5px;cursor:pointer;font:inherit}button:disabled{opacity:.4}p{line-height:1.5}small{color:#637068}#fin{display:none}</style>\n<h1>¿Cuál es el original?</h1><p>Compara A y B. Elige el que crees que pertenece al plano original. Tu respuesta queda guardada en este navegador; al terminar descarga el archivo para el custodio.</p><p id="progreso"></p><div id="pares"><div class="opcion"><h2>A</h2><img id="a" alt="Opción A"><button id="eligeA">Elegir A</button></div><div class="opcion"><h2>B</h2><img id="b" alt="Opción B"><button id="eligeB">Elegir B</button></div></div><p id="fin">Respuestas completas. Descarga el archivo y entrégalo al custodio. Esta página no contiene la clave.</p><p><button id="descargar">Descargar respuestas</button></p><small>Prueba de desarrollo · no acredita por sí sola el objetivo de indistinguibilidad. No reinicies ni repitas para cambiar respuestas.</small>\n<script>\nconst data=__DATOS__,key=\'letra-viva-ab-\'+data.sesion;let answers=JSON.parse(localStorage.getItem(key)||\'[]\');\nfunction draw(){const n=answers.length;document.getElementById(\'progreso\').textContent=n+\' de \'+data.pares.length+\' respuestas\';if(n>=data.pares.length){document.getElementById(\'pares\').style.display=\'none\';document.getElementById(\'fin\').style.display=\'block\';return;}const p=data.pares[n];document.getElementById(\'a\').src=p.A;document.getElementById(\'b\').src=p.B;}\nfunction answer(choice){const p=data.pares[answers.length];if(!p)return;answers.push({id:p.id,eleccion:choice,fecha:new Date().toISOString()});localStorage.setItem(key,JSON.stringify(answers));draw();}\ndocument.getElementById(\'eligeA\').onclick=()=>answer(\'A\');document.getElementById(\'eligeB\').onclick=()=>answer(\'B\');\ndocument.getElementById(\'descargar\').onclick=()=>{const blob=new Blob([JSON.stringify({sesion:data.sesion,respuestas:answers},null,2)],{type:\'application/json\'});const a=document.createElement(\'a\');a.href=URL.createObjectURL(blob);a.download=\'respuestas-ab.json\';a.click();setTimeout(()=>URL.revokeObjectURL(a.href),1000);};draw();\n</script></html>\n', '__main__.py': '"""Laboratorio D-12. Todas las salidas privadas deben estar fuera del repositorio."""\nimport argparse\nimport json\nfrom pathlib import Path\nfrom .fuente import construir, completar_mayuscula\nfrom .editar import editar\nfrom .restaurar import restaurar\nfrom .ab import preparar, evaluar\n\n\ndef main():\n    ap=argparse.ArgumentParser(description=__doc__)\n    sub=ap.add_subparsers(dest=\'accion\',required=True)\n    f=sub.add_parser(\'fuente\');f.add_argument(\'pdf\');f.add_argument(\'adn\');f.add_argument(\'salida\')\n    f.add_argument(\'--propio\',action=\'store_true\');f.add_argument(\'--excluir-span\',type=int,action=\'append\',default=[])\n    f.add_argument(\'--pagina\',type=int,default=1);f.add_argument(\'--fuente-xref\',type=int)\n    c=sub.add_parser(\'completar\');c.add_argument(\'fuente\');c.add_argument(\'salida\')\n    c.add_argument(\'--destino\',required=True);c.add_argument(\'--modelo\',required=True);c.add_argument(\'--altura\',required=True)\n    e=sub.add_parser(\'editar\');e.add_argument(\'pdf\');e.add_argument(\'adn\');e.add_argument(\'fuente\');e.add_argument(\'salida\')\n    e.add_argument(\'--propio\',action=\'store_true\');e.add_argument(\'--pagina\',type=int,default=1)\n    e.add_argument(\'--span\',type=int,required=True);e.add_argument(\'--texto\')\n    e.add_argument(\'--region\',nargs=4,type=float,required=True)\n    e.add_argument(\'--excluir-region\',nargs=4,type=float,action=\'append\',default=[])\n    e.add_argument(\'--metodo\',choices=[\'simple\',\'estilo\'],default=\'simple\')\n    r=sub.add_parser(\'restaurar\');r.add_argument(\'edicion\');r.add_argument(\'salida\')\n    b=sub.add_parser(\'ab\');b.add_argument(\'pares_json\');b.add_argument(\'participante\');b.add_argument(\'custodio\')\n    v=sub.add_parser(\'evaluar-ab\');v.add_argument(\'respuestas\');v.add_argument(\'clave\')\n    a=ap.parse_args()\n    if a.accion==\'fuente\':\n        result=construir(a.pdf,a.adn,a.salida,propio=a.propio,pagina=a.pagina,excluir_spans=a.excluir_span,fuente_xref=a.fuente_xref)\n    elif a.accion==\'completar\':\n        result=completar_mayuscula(a.fuente,a.salida,destino=a.destino,modelo_minuscula=a.modelo,referencia_altura=a.altura)\n    elif a.accion==\'editar\':\n        result=editar(a.pdf,a.adn,a.fuente,a.salida,propio=a.propio,pagina=a.pagina,span_id=a.span,texto=a.texto,\n                      autorizado=a.region,excluidas=a.excluir_region,metodo=a.metodo)\n    elif a.accion==\'restaurar\':result=restaurar(a.edicion,a.salida)\n    elif a.accion==\'ab\':result=preparar(json.loads(Path(a.pares_json).read_text()),a.participante,a.custodio)\n    else:result=evaluar(a.respuestas,a.clave)\n    print(json.dumps(result,ensure_ascii=False,indent=2))\n\n\nif __name__==\'__main__\':main()\n', 'test_letra_viva.py': "import json\nimport tempfile\nimport unittest\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image\nfrom fontTools.fontBuilder import FontBuilder\nfrom fontTools.pens.ttGlyphPen import TTGlyphPen\nfrom fontTools.ttLib import TTFont\nfrom .fuente import contorno,medoid,componer_faltante,completar_mayuscula\nfrom .editar import reponer_region,dentro,intersecta\nfrom .ab import preparar,evaluar\nfrom .restaurar import restaurar\nfrom .comun import sha256,abrir_adn\n\n\nclass LetraTest(unittest.TestCase):\n    def test_medoid_elige_muestra_central_real(self):\n        a=np.zeros((10,10),np.uint8);b=a.copy();b[3:7,3:7]=255;c=b.copy();c[3,3]=0\n        self.assertIn(medoid([dict(mask=x) for x in (a,b,c)]),(1,2))\n\n    def test_contorno_con_hueco_se_conserva(self):\n        mask=np.zeros((40,40),np.uint8);mask[5:35,5:35]=255;mask[12:28,12:28]=0\n        glyph=contorno(mask,40,35)\n        self.assertEqual(glyph.numberOfContours,2)\n        ends=glyph.endPtsOfContours;coords=np.array(glyph.coordinates);areas=[];start=0\n        for end in ends:\n            p=coords[start:end+1];areas.append(np.sum(p[:,0]*np.roll(p[:,1],-1)-p[:,1]*np.roll(p[:,0],-1)));start=end+1\n        self.assertLess(areas[0]*areas[1],0)\n\n    def test_solo_region_cambia_pixel_a_pixel(self):\n        a=np.zeros((30,30,3),np.uint8);b=np.full_like(a,255)\n        out=reponer_region(a,b,(5,6,10,12));self.assertEqual(int(np.any(out!=a,axis=2).sum()),30)\n        out[6:12,5:10]=0;np.testing.assert_array_equal(a,out)\n\n    def test_zonas_excluidas_y_contencion(self):\n        self.assertTrue(intersecta([0,0,10,10],[9,9,12,12]))\n        self.assertFalse(intersecta([0,0,10,10],[10,0,20,10]))\n        self.assertFalse(dentro([0,0,10,10],[0,0,11,10]))\n\n    def test_baseline_union_sin_entrenamiento(self):\n        a=np.eye(10,dtype=np.uint8)*255;b=np.fliplr(a)\n        out=componer_faltante(a,b,'union');self.assertEqual(np.count_nonzero(out),20)\n        with self.assertRaises(ValueError):componer_faltante(a,b[:5],'union')\n\n    def test_ttf_completada_reabre_sin_duplicar_glyphorder(self):\n        with tempfile.TemporaryDirectory() as td:\n            root=Path(td);mask=np.eye(40,dtype=np.uint8)*255\n            # Dos formas propias con área para comprobar contenedores, no estética.\n            mask[10:30,10:30]=255;g=contorno(mask,40,35)\n            fb=FontBuilder(1000,isTTF=True);fb.setupGlyphOrder(['.notdef','x','A'])\n            fb.setupCharacterMap({120:'x',65:'A'});fb.setupGlyf({'.notdef':TTGlyphPen(None).glyph(),'x':g,'A':g})\n            fb.setupHorizontalMetrics({n:(800,0) for n in ['.notdef','x','A']});fb.setupHorizontalHeader(ascent=1000,descent=-200)\n            fb.setupNameTable(dict(familyName='Fixture',styleName='Regular'));fb.setupOS2();fb.setupPost();fb.setupMaxp();fb.save(root/'a.ttf')\n            completar_mayuscula(root/'a.ttf',root/'b.ttf',destino='X',modelo_minuscula='x',referencia_altura='A')\n            font=TTFont(root/'b.ttf');self.assertIn(ord('X'),font.getBestCmap());self.assertEqual(len(font.getGlyphOrder()),4)\n\n    def test_ab_clave_separada_sin_respuestas_falsas(self):\n        with tempfile.TemporaryDirectory() as td:\n            root=Path(td);Image.new('RGB',(20,20),'white').save(root/'a.png');Image.new('RGB',(20,20),'gray').save(root/'b.png')\n            r=preparar([dict(original=root/'a.png',rehecha=root/'b.png',sha256_origen='a'*64)],root/'participante',root/'custodio')\n            self.assertEqual(r['respuestas'],0);self.assertFalse((root/'participante/clave.json').exists())\n            public=json.loads((root/'participante/pares.json').read_text());self.assertNotIn('original',public['pares'][0])\n            (root/'respuestas.json').write_text(json.dumps(dict(sesion=r['sesion'],respuestas=[])))\n            self.assertIsNone(evaluar(root/'respuestas.json',root/'custodio/clave.json')['acierto'])\n\n    def test_restauracion_exacta_y_rechazo_de_alteracion(self):\n        with tempfile.TemporaryDirectory() as td:\n            root=Path(td);original=np.zeros((20,20,3),np.uint8);after=original.copy();after[2:5,3:8]=255\n            Image.fromarray(original).save(root/'pagina-antes.png');Image.fromarray(after).save(root/'pagina-despues.png')\n            Image.fromarray(original[2:5,3:8]).save(root/'parche-antes.png')\n            r=dict(region_px=[3,2,8,5],sha256_antes=sha256(root/'pagina-antes.png'),\n                   sha256_despues=sha256(root/'pagina-despues.png'),sha256_parche_antes=sha256(root/'parche-antes.png'))\n            (root/'edicion.json').write_text(json.dumps(r))\n            self.assertTrue(restaurar(root,root/'restaurada.png')['pixel_identico'])\n            Image.new('RGB',(5,3),'red').save(root/'parche-antes.png')\n            with self.assertRaisesRegex(ValueError,'alterados'):restaurar(root,root/'fallo.png')\n            self.assertFalse((root/'fallo.png').exists())\n\n    def test_adn_de_otro_origen_rechazado(self):\n        with tempfile.TemporaryDirectory() as td:\n            root=Path(td);(root/'doc').write_bytes(b'propio');(root/'adn.json').write_text(json.dumps(dict(sha256_origen='0'*64)))\n            with self.assertRaisesRegex(ValueError,'no corresponde'):abrir_adn(root/'doc',root/'adn.json')\n\n    def test_ab_no_acepta_doble_voto(self):\n        with tempfile.TemporaryDirectory() as td:\n            root=Path(td);(root/'key.json').write_text(json.dumps(dict(sesion='s',pares=[dict(id='1',original='A')])))\n            (root/'r.json').write_text(json.dumps(dict(sesion='s',respuestas=[dict(id='1',eleccion='A')]*2)))\n            with self.assertRaisesRegex(ValueError,'duplicada'):evaluar(root/'r.json',root/'key.json')\n\n\nif __name__=='__main__':unittest.main()\n", 'PRERREGISTRO.md': '# D-12 · Prerregistro v1 · 26-10-2026\n\nANTES de métricas y entrenamiento. Semilla 2610. Estable D-10 permanece\nseparado; solo tools/letra_viva. Un proceso GPU a la vez, únicamente Colab\nde Bruno, saldo compartido con D-10 <=20 unidades. No se autoriza descargar\npesos por este protocolo; se requiere licencia/procedencia y permiso vigente.\n\n## Pregunta\n\n¿Una lección de estilo del documento permite reconstruir palabras nuevas\nmejor que padre, maestro y la combinación simple de muestras/trazos? No se\nconfunde recuperación de memoria externa (glifos) con entrenamiento de pesos.\n\nEl plano de desarrollo es de autoría compartida confirmada por el usuario:\nSHA ff05702c3fe241d53bcab7fdcd37a925bb619e54313b6074bdedc68e32e9b1a9.\nUna página y un origen, ya vistos por el constructor. SOLO DESARROLLO: no se\npuede obtener una reserva independiente repartiendo recortes de esa página.\nCustodio distinto (Claude/Bruno) debe aportar otros documentos propios o\nejemplos de Prometeo, separar por SHA de origen y sellar DESPUÉS del candidato.\nEste constructor no abre esa reserva ni d10-examen. Copias/transformaciones\ndel plano son una evidencia, no réplicas independientes.\n\n## Brazos y orden\n\nPrimero padre sin lección y baseline: fuente nativa reutilizable cuando su\nmapeo Unicode esté disponible; si no, fuente derivada de muestras observadas\ny medoid por carácter. Missing glyph baseline: composición explícita de\ntrazos/glifos observados, nunca alfabeto inventado ni fallback silencioso.\nSin regla de composición o muestras suficientes, abstenerse y contar cobertura.\nMaestro: origen nativo A, informa geometría y tinta; no juzga la reserva.\nAlumno generativo: aún no ejecutable hasta resolver código/pesos compatibles,\ncorpus de entrenamiento autorizado y presupuesto. Placebo: lección permutada\nde igual tamaño/arquitectura/coste; sin éste no atribuir mejora al maestro.\nSi padre ya llega al techo o al suelo, parar y rediseñar con desarrollo.\n\n## Rehacer y medir\n\nSolo regiones explícitas autorizadas, jamás firma/sello ni marca educativa.\nComprobar que TODO píxel fuera del rectángulo autorizado es idéntico. Mantener\nhistorial con origen, región, parámetros y hashes antes/después, más parches\nreversibles locales. Un editor raster no afirma conservar objetos PDF.\nSe mide SSIM RGB (data_range=255) del recorte, MAE y cobertura; LPIPS solo con\npesos aprobados y verificados, de lo contrario null/NO VERIFICADO. No sustituir\nLPIPS por una red aleatoria ni inventar cifras. Registrar DPI, tiempo y coste.\nMedir tinta Lab, tamaño, inclinación, textura y residual de línea base;\nsin certeza suficiente, informar limitación o abstenerse.\n\nEn desarrollo: regenerar texto de spans horizontales del mismo plano, pero\nexcluir todas sus cajas de las muestras que construyen la fuente. No entrenar\ncon esas cajas. Como hay un solo origen, sigue siendo desarrollo intrasource,\nno transferencia ni holdout independiente. Caso fijo inicial: span 18 del ADN\nnativo (índice desde cero). No elegir después el recorte que mejor salga.\nComparar baseline de fondo plano y escritura simple vs inpainting y estilo\nmedido, ambos con la MISMA fuente y la MISMA región. No adjudicar ventaja a\nun alumno generativo que no se ha ejecutado.\n\n## A/B ciega\n\nEl generador mezcla pares original/rehecho con semilla fijada y guarda clave\nseparada para custodio. La interfaz del participante no recibe la clave;\nrespuestas persistentes y rechazadas si se repiten. No se rellenan por Bruno.\nEn la reserva nueva, mínimo 100 palabras y >=10 páginas, agrupación por origen.\nMeta puntual <=55% de identificación humana; reportar intervalo Wilson 95%\ny n (con agrupación por origen al interpretar). No afirmar indistinguibilidad\nestadística con pocos ejemplos o IC compatible con >55%. El caso de desarrollo\nsolo prueba el mecanismo de presentación y respuesta, no el objetivo A/B.\n\n## Muerte, regresiones, recuperación y FICHA\n\nNo promover si cobertura <1 en población prerregistrada, altera un píxel\nexterno, no mejora baseline, falla integridad, no hay ventaja frente a placebo\no no cumple A/B en reserva independiente. Guardar negativos y abstenciones.\nDespués de cada ronda medir tarea anterior para olvido, con checkpoint a Drive\nantes de continuar. Sin GPU/corpus/licencia: registrar bloqueo y experimento\nbarato en SEGUNDA-MIRADA.md, no afirmar que la capacidad no existe.\n\nFICHA D12-2610; productor Codex C3; experiencia: rehacer texto propio sin\nplantilla; maestro A; estudiante generativo pendiente; evaluación de estudiante\npendiente custodio; información heredable: muestras/contornos, procedencia y\neventuales pesos por separado. Beneficio/regresiones/transferencia: por medir.\nCaduca al cambiar fuente, extracción, origen, presupuesto, licencia o método.\nPesos entrenados: ninguno. No se integran ni duplican núcleos FORGE252 A/B/C.\n', 'LICENCIAS.json': '{\n  "fecha": "2026-10-26",\n  "modelos": [\n    {\n      "modelo": "MX-Font",\n      "revision": "93f3c88517f7c904f16da6333adb2588dcdf3cce",\n      "evidencia": [\n        {\n          "url": "https://raw.githubusercontent.com/clovaai/mxfont/93f3c88517f7c904f16da6333adb2588dcdf3cce/readme.md",\n          "sha256_texto_recuperado": "f7d4c8ac9e33a22875943ecd4eba9cf485777ad6bf10033280e1e3a3553e410e"\n        },\n        {\n          "url": "https://raw.githubusercontent.com/clovaai/mxfont/93f3c88517f7c904f16da6333adb2588dcdf3cce/LICENSE",\n          "sha256_texto_recuperado": "61d296a4046a78db8276424eb866089e6486cf288da1a92d2f7397308a93b9a3"\n        },\n        {\n          "url": "https://raw.githubusercontent.com/clovaai/mxfont/93f3c88517f7c904f16da6333adb2588dcdf3cce/NOTICE",\n          "sha256_texto_recuperado": "f4886a3a2754d2b9a777dce1fc3cbe68b6cbe3ee5be1995d91b4c56cf6727457"\n        }\n      ],\n      "decision": "MIT general; excepción FUNIT CC-BY-NC-SA-4.0 en NOTICE. No adoptado.",\n      "pesos_descargados": false,\n      "sha256_pesos": null,\n      "licencia_pesos": "NO RESUELTA",\n      "instalado": false\n    },\n    {\n      "modelo": "FontDiffuser",\n      "revision": "7b28ce9c3b357f4fb23296622f458cf169803539",\n      "evidencia": [\n        {\n          "url": "https://raw.githubusercontent.com/yeungchenwa/FontDiffuser/7b28ce9c3b357f4fb23296622f458cf169803539/README.md",\n          "sha256_texto_recuperado": "ec1e22548415d122e5d5c2f11ac931ac5085de6d512c0fd9a63693ddf0f58b7c"\n        }\n      ],\n      "decision": "Uso limitado a investigación no comercial; no adoptado.",\n      "pesos_descargados": false,\n      "sha256_pesos": null,\n      "licencia_pesos": "NO RESUELTA",\n      "instalado": false\n    }\n  ],\n  "fonttools": {\n    "version": "4.60.1",\n    "licencia": "MIT",\n    "fuente": "https://pypi.org/project/fonttools/4.60.1/"\n  },\n  "pymupdf": {\n    "version": "1.26.5",\n    "licencia": "AGPL/comercial",\n    "estado": "Preexistente; reutilizado, no instalado en C3; distribución no evaluada"\n  }\n}\n', 'requirements-c3.txt': '# Nuevas dependencias del venv C3. No ejecutar contra Python sistema.\n# Python 3.9.6; torch 2.8.0 se reutiliza sin actualizarlo.\nfonttools==4.60.1\nscikit-image==0.24.0\nscipy==1.13.1\nimageio==2.37.2\nlazy-loader==0.6\ntifffile==2024.8.30\npyspellchecker==0.8.3\n# Ya disponibles: numpy 1.26.4, Pillow 11.3.0, opencv-python 4.11.0.86,\n# PyMuPDF 1.26.5 (AGPL/comercial, preexistente: no se instala con esta lista).\n# requirements no instala ni contiene pesos ni fuentes.\n'}
HASHES = {'__init__.py': 'b8216056668c4c3123e419039f5146cf6ef47649bc79c62313350fd7f5ebadf3', 'comun.py': '98b66980a701c53435fa109e31920f54aeda8c99ad948be12a98c4d32f6d2053', 'fuente.py': '46bef2b065058fa583560db2e23fa7d0e243f6aaffe52f6c9c88c2dc6f9a5f03', 'editar.py': 'fd303732440d77f2fece5802f898a527218ba053ec4b77600c9d6b9b98762acf', 'restaurar.py': '394fc034f1c47cf0dd660ee08dac63b1d884b78352b715572ce4deab26df6927', 'ab.py': '379bf51b388b2f98e15fe9b025e55e0861c9fa8b8a38768c6a7d148f7e314818', 'ab.html': 'a5d9d03bc1298465f8c63c87fca50ca0fddb9faccb40811b25df15daa371a2ff', '__main__.py': '162db6f7dac6c674d876b44877b15b138a5b89ca50f474c8a59cb69022628bd7', 'test_letra_viva.py': 'ce6cac4d70d3b14af70b2b90b7ceb5e3c305ee7ef351d7ae1431aca8847ffeee', 'PRERREGISTRO.md': '59d9bbace5e3aee1e65ec87a1b521ae6c7946834b5abc71c669a35feeec8a4af', 'LICENCIAS.json': '63eeaa01f68270c2cb80010dc4e1cd4a185824c52b547e8d37729adc9dcf91ab', 'requirements-c3.txt': 'c61850cf965df73259991d64047cebb9c42034d3164607a27ffde614bfc76ae2'}
for name, source in PAYLOAD.items():
    assert hashlib.sha256(source.encode()).hexdigest() == HASHES[name]
    target = ROOT / "tools" / "letra_viva" / name
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and target.read_text() != source:
        raise RuntimeError("Salida de código existente distinta; conservar antes de continuar")
    target.write_text(source)
sys.path.insert(0,str(ROOT))
print(json.dumps(HASHES,indent=2))


## Entorno

Reutilizar un entorno autorizado con PyMuPDF, NumPy, Pillow, OpenCV, scikit-image y fontTools. No se instala PyMuPDF AGPL/comercial ni se cambia torch. Si falta algo, detener y resolver el entorno; la autorización vigente en la Mac solo cubre su venv C3. Ejecutar en CPU. La celda siguiente solo verifica importaciones.

In [ ]:
import pymupdf, numpy, PIL, cv2, skimage, fontTools
import unittest
suite=unittest.defaultTestLoader.loadTestsFromName("tools.letra_viva.test_letra_viva")
assert unittest.TextTestRunner(verbosity=2).run(suite).wasSuccessful()


## Drive y documento de desarrollo

Bruno configura rutas y revisa la región. El SHA fijado es el plano ya autorizado, sin reservas. No cargar documentos ajenos ni firmados/sellados. Las coordenadas de ejemplo corresponden únicamente a ese SHA. Cada ejecución crea una carpeta nueva en Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from datetime import datetime, timezone
from tools.letra_viva.comun import sha256, guardar
BASE=Path("/content/drive/MyDrive/Prometeo/LetraViva")
PDF=BASE/"entrada/plano-demo.pdf"
ADN=BASE/"entrada/plano-demo.adn.json"
ES_PROPIO=False  # Bruno confirma su autoría y revisión de exclusiones
assert ES_PROPIO, "Confirmar documento propio antes de procesar"
assert sha256(PDF)=="ff05702c3fe241d53bcab7fdcd37a925bb619e54313b6074bdedc68e32e9b1a9"
RUN=BASE/"salidas"/datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN.mkdir(parents=True,exist_ok=False)
REGION=[1801.19384765625,251.4849853515625,1887.8385009765625,276.5003662109375]
EXCLUIDAS=[]  # Definir firmas/sellos rasterizados y otras zonas intocables


In [ ]:
from tools.letra_viva.fuente import construir, completar_mayuscula
from fontTools.ttLib import TTFont
report=construir(PDF,ADN,RUN/"fuente",excluir_spans=[18],propio=ES_PROPIO)
FONT=RUN/"fuente"/report["fuente"]
with TTFont(FONT) as ft:
    cmap=ft.getBestCmap()
if ord("X") not in cmap:
    completar_mayuscula(FONT,RUN/"baseline-completa.ttf",destino="X",modelo_minuscula="x",referencia_altura="A")
    FONT=RUN/"baseline-completa.ttf"
print("FUENTE SHA256",sha256(FONT))


In [ ]:
from tools.letra_viva.editar import editar
from tools.letra_viva.restaurar import restaurar
from tools.letra_viva.ab import preparar
results={}
for method in ("simple","estilo"):
    results[method]=editar(PDF,ADN,FONT,RUN/method,propio=ES_PROPIO,span_id=18,autorizado=REGION,excluidas=EXCLUIDAS,metodo=method)
restaurar(RUN/"estilo",RUN/"restaurada.png")
ab=preparar([dict(original=RUN/"estilo/parche-antes.png",rehecha=RUN/"estilo/parche-despues.png",sha256_origen=sha256(PDF))],RUN/"PARTICIPANTE",RUN/"CUSTODIO")
guardar(RUN/"comparacion.json",dict(resultados=results,ab=ab,gpu_unidades=0,entrenamiento=False,dominio="DESARROLLO"))
manifest={str(p.relative_to(RUN)):sha256(p) for p in sorted(RUN.rglob("*")) if p.is_file()}
guardar(RUN/"manifest.json",manifest)
print(json.dumps({m:r["metricas"] for m,r in results.items()},indent=2))
print("MANIFEST SHA256",sha256(RUN/"manifest.json"))


## Entrenamiento pendiente — no ejecutar como si existiera

FontDiffuser: restricción no comercial. MX-Font: MIT con excepción FUNIT CC-BY-NC-SA. Los SHA de evidencia están en LICENCIAS.json; pesos SHA=null porque no se descargaron. Falta seleccionar una vía compatible y autorizar el artefacto exacto. No hay curva, checkpoint neuronal ni mejora generativa. La baseline perdedora se conserva. Custodio independiente aporta reserva nueva después del congelado. No usar la reserva como entrenamiento.

In [ ]:
estado=dict(fase="D-12",estado="BLOQUEADA",pesos_sha256=None,entrenador_implementado=False,
    faltantes=["licencias de código/pesos compatibles", "permiso de descarga de pesos",
               "corpus autorizado", "reserva independiente", "saldo Colab compartido"],
    gpu_unidades_consumidas=0)
print(json.dumps(estado,ensure_ascii=False,indent=2))
guardar(RUN/"bloqueo-entrenamiento.json",estado)
